In [1]:
%pip install pymorphy3 pymorphy3-dicts-ru

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Оффтоп
Так как в прошлый раз забыл что мы решили классифицировать по 3-4 темам, теперь сделал маппинг на 4 темы
Вроде должно теперь быстро отрабатывать, у меня за 20 минут все отрабатывает

Предобработка

In [ ]:
from pathlib import Path
import re
import multiprocessing
import numpy as np
import pandas as pd
import pymorphy3
from sklearn.model_selection import train_test_split
from joblib import Parallel, delayed

DATA_PATH = Path("/content/lenta-ru-news.csv") #Надеюсь ты не забудешь сменить путь к файлу)
RANDOM_STATE = 42

MAX_CHARS     = 1200
MIN_TEXT_LEN  = 60
MAX_PER_CLASS = 30_000

# Тут можно поменять маппинг, если изменить в "Интернет и СМИ" "Технологии" на "Новости", 
# то небольшая часть ошибок пропадет, я пробовал и так и так разница минимальная, поэтому оставил как есть
# Если захочешь можешь поменять
TOPIC_MAP = {
    "Культура":          "Развлечения",
    "Ценности":          "Развлечения",
    "Культпросвет":      "Развлечения",
    "Спорт":             "Спорт",
    "Наука и техника":   "Технологии",
    "Интернет и СМИ":    "Технологии", # <- Вот тут "Новости"
    "Россия":            "Новости",
    "Мир":               "Новости",
    "Бывший СССР":       "Новости",
    "Экономика":         "Новости",
    "Бизнес":            "Новости",
    "Силовые структуры": "Новости",
    "Из жизни":          "Новости",
    "Дом":               "Новости",
    "Путешествия":       "Новости",
    "69-я параллель":    "Новости",
    "Крым":              "Новости",
}
TOPICS_KEEP = set(TOPIC_MAP.keys())

_re_html  = re.compile(r"<[^>]+>")
_re_url   = re.compile(r"https?://\S+|www\.\S+")
_re_mail  = re.compile(r"\S+@\S+")
_re_nonal = re.compile(r"[^а-яa-z\s]")
_re_space = re.compile(r"\s+")

_MORPH = pymorphy3.MorphAnalyzer()
_LEMMA_CACHE = {}


def _lemma(token: str) -> str:
    cached = _LEMMA_CACHE.get(token)
    if cached is not None:
        return cached
    lemma = _MORPH.parse(token)[0].normal_form
    _LEMMA_CACHE[token] = lemma
    return lemma


def clean_and_lemmatize(value: str) -> str:
    text = "" if value is None else str(value)
    if len(text) > MAX_CHARS:
        text = text[:MAX_CHARS]
    text = _re_html.sub(" ", text)
    text = _re_url.sub(" ", text)
    text = _re_mail.sub(" ", text)
    text = text.lower().replace("ё", "е")
    text = _re_nonal.sub(" ", text)
    text = _re_space.sub(" ", text).strip()
    if not text:
        return ""
    tokens = [_lemma(t) for t in text.split()]
    return " ".join(t for t in tokens if len(t) > 1)


# Загрузка
parts = []
for chunk in pd.read_csv(
    DATA_PATH,
    usecols=["title", "text", "topic"],
    dtype=str,
    encoding="utf-8",
    keep_default_na=False,
    chunksize=100_000,
    on_bad_lines="error",
):
    chunk["topic"] = chunk["topic"].str.strip()
    chunk = chunk[chunk["topic"].isin(TOPICS_KEEP)]
    if not chunk.empty:
        parts.append(chunk.copy())

df = pd.concat(parts, ignore_index=True)
print("Загружено записей нужных рубрик:", len(df))

before = len(df)
df = df.drop_duplicates(subset=["title", "text"]).reset_index(drop=True)
print(f"Удалено точных дублей (raw): {before - len(df)}")

# Маппинг
df["category"] = df["topic"].map(TOPIC_MAP)
df = df[df["category"].notna()].copy()
print("\nРаспределение по 4 темам:")
print(df["category"].value_counts())

# Балансировка
df = (
    df.sample(frac=1, random_state=RANDOM_STATE)
      .groupby("category", group_keys=False, sort=False)
      .head(MAX_PER_CLASS)
      .reset_index(drop=True)
)
print("\nПосле балансировки:")
print(df["category"].value_counts())

df["raw_full"] = (
    df["title"].fillna("") + ". " + df["text"].fillna("")
).str.slice(0, MAX_CHARS + 200)

# Лемматизация
N_JOBS = min(4, max(1, multiprocessing.cpu_count() - 1))
print("\nВоркеров:", N_JOBS)

cleaned = Parallel(
    n_jobs=N_JOBS, backend="threading", batch_size=500, verbose=5,
)(
    delayed(clean_and_lemmatize)(t) for t in df["raw_full"].tolist()
)
df["text_clean"] = cleaned

df = df[df["text_clean"].str.len() >= MIN_TEXT_LEN].copy()

before = len(df)
df = df.drop_duplicates(subset=["text_clean"]).reset_index(drop=True)
print(f"\nУдалено дубликатов по text_clean: {before - len(df)}")
print("Осталось записей:", len(df))

X = df["text_clean"].to_numpy()
y = df["category"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y,
)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=RANDOM_STATE, stratify=y_train,
)
print("\nTrain / Val / Test:", X_tr.shape, X_val.shape, X_test.shape)

Загружено записей нужных рубрик: 738787
Удалено точных дублей (raw): 19

Распределение по 4 темам:
category
Новости        514662
Технологии      97798
Спорт           64411
Развлечения     61897
Name: count, dtype: int64

После балансировки:
category
Новости        30000
Технологии     30000
Спорт          30000
Развлечения    30000
Name: count, dtype: int64

Воркеров: 4


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done 5008 tasks      | elapsed:    8.3s
[Parallel(n_jobs=4)]: Done 32008 tasks      | elapsed:   24.9s
[Parallel(n_jobs=4)]: Done 77008 tasks      | elapsed:   45.4s
[Parallel(n_jobs=4)]: Done 120000 out of 120000 | elapsed:  1.0min finished



Удалено дубликатов по text_clean: 0
Осталось записей: 119927

Train / Val / Test: (81549,) (14392,) (23986,)


Векторизация TF-IDF и подбор гиперпараметра C на валидации

In [3]:
import gc
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Векторизация
features = FeatureUnion([
    ("word", TfidfVectorizer(
        analyzer="word", ngram_range=(1, 2),
        min_df=3, max_df=0.9, sublinear_tf=True,
        max_features=200_000, dtype=np.float32,
    )),
    ("char", TfidfVectorizer(
        analyzer="char_wb", ngram_range=(3, 5),
        min_df=5, max_df=0.9, sublinear_tf=True,
        max_features=100_000, dtype=np.float32,
    )),
])

print("Считаю TF-IDF на train...")
X_tr_vec = features.fit_transform(X_tr)
print("Train:", X_tr_vec.shape, f"{X_tr_vec.data.nbytes/1e6:.0f} MB")

print("Считаю TF-IDF на val...")
X_val_vec = features.transform(X_val)
print("Val:  ", X_val_vec.shape)

print("Считаю TF-IDF на test...")
X_test_vec = features.transform(X_test)
print("Test: ", X_test_vec.shape)
gc.collect()


# Подбор C для LinearSVC на val
print("\nПодбор C на val для LinearSVC:")
svc_scores = {}
for C in [0.03, 0.1, 0.3, 1.0, 3.0]:
    clf = LinearSVC(
        C=C, class_weight="balanced", dual="auto",
        random_state=RANDOM_STATE, max_iter=15_000,
    )
    clf.fit(X_tr_vec, y_tr)
    score = f1_score(y_val, clf.predict(X_val_vec), average="macro")
    svc_scores[C] = score
    print(f"  C={C:>5}  val macro F1 = {score:.4f}")

best_C = max(svc_scores, key=svc_scores.get)
best_score = svc_scores[best_C]
print(f"\nЛучший C для SVC: {best_C} | val macro F1: {best_score:.4f}")


# Подбор C для LogisticRegression на val
print("\nПодбор C на val для LogisticRegression:")
lr_scores = {}
for C in [0.1, 0.3, 1.0, 3.0, 10.0]:
    clf = LogisticRegression(
        C=C, class_weight="balanced", solver="lbfgs",
        max_iter=2_000, tol=1e-4,
        random_state=RANDOM_STATE,
    )
    clf.fit(X_tr_vec, y_tr)
    score = f1_score(y_val, clf.predict(X_val_vec), average="macro")
    lr_scores[C] = score
    print(f"  C={C:>5}  val macro F1 = {score:.4f}")

best_C_lr = max(lr_scores, key=lr_scores.get)
best_score_lr = lr_scores[best_C_lr]
print(f"\nЛучший C для LogReg: {best_C_lr} | val macro F1: {best_score_lr:.4f}")


# Финальная модель + оценка на test
if best_score_lr > best_score:
    print(f"\nНа val побеждает LogisticRegression (C={best_C_lr}).")
    best_model = LogisticRegression(
        C=best_C_lr, class_weight="balanced", solver="lbfgs",
        max_iter=2_000, tol=1e-4,
        random_state=RANDOM_STATE,
    )
    best_model_name = "LogReg"
else:
    print(f"\nНа val побеждает LinearSVC (C={best_C}).")
    best_model = LinearSVC(
        C=best_C, class_weight="balanced", dual="auto",
        random_state=RANDOM_STATE, max_iter=15_000,
    )
    best_model_name = "LinearSVC"

best_model.fit(X_tr_vec, y_tr)
pred_svc = best_model.predict(X_test_vec)

print(f"\n=== {best_model_name} (best C) ===")
print("Accuracy:", round(accuracy_score(y_test, pred_svc), 4))
print("Macro F1:", round(f1_score(y_test, pred_svc, average="macro"), 4))
print("Weighted F1:", round(f1_score(y_test, pred_svc, average="weighted"), 4))
print()
print(classification_report(y_test, pred_svc, zero_division=0))
gc.collect()

Считаю TF-IDF на train...
Train: (81549, 300000) 478 MB
Считаю TF-IDF на val...
Val:   (14392, 300000)
Считаю TF-IDF на test...
Test:  (23986, 300000)

Подбор C на val для LinearSVC:
  C= 0.03  val macro F1 = 0.9306
  C=  0.1  val macro F1 = 0.9369
  C=  0.3  val macro F1 = 0.9388
  C=  1.0  val macro F1 = 0.9383
  C=  3.0  val macro F1 = 0.9366

Лучший C для SVC: 0.3 | val macro F1: 0.9388

Подбор C на val для LogisticRegression:
  C=  0.1  val macro F1 = 0.9220
  C=  0.3  val macro F1 = 0.9296
  C=  1.0  val macro F1 = 0.9344
  C=  3.0  val macro F1 = 0.9361
  C= 10.0  val macro F1 = 0.9357

Лучший C для LogReg: 3.0 | val macro F1: 0.9361

На val побеждает LinearSVC (C=0.3).

=== LinearSVC (best C) ===
Accuracy: 0.9399
Macro F1: 0.9398
Weighted F1: 0.9399

              precision    recall  f1-score   support

     Новости       0.91      0.91      0.91      5996
 Развлечения       0.95      0.96      0.95      6000
       Спорт       0.99      0.99      0.99      5999
  Технологии  

30

Анализ ошибок на тестовой выборке

In [4]:
import pandas as pd

err_df = pd.DataFrame({"text": X_test, "true": y_test, "pred": pred_svc})
errors = err_df[err_df["true"] != err_df["pred"]].copy()

print("Всего ошибок:", len(errors), f"({len(errors)/len(err_df)*100:.2f}%)")

confusions = errors.groupby(["true", "pred"]).size().sort_values(ascending=False)
print("\nТоп-20 смешений (истина -> предсказание):")
print(confusions.head(20))

for (true_label, pred_label), count in confusions.head(8).items():
    print(f"\n=== {true_label} -> {pred_label}: {count} ===")
    sample = errors[(errors["true"] == true_label) &
                    (errors["pred"] == pred_label)].head(3)
    for _, row in sample.iterrows():
        print("\nТекст:", " ".join(row["text"].split())[:400])
        print("Истина:", row["true"], "| Предсказано:", row["pred"])

Всего ошибок: 1441 (6.01%)

Топ-20 смешений (истина -> предсказание):
true         pred       
Новости      Технологии     377
Технологии   Новости        372
Новости      Развлечения    151
Технологии   Развлечения    142
Развлечения  Новости        130
             Технологии     106
Новости      Спорт           39
Спорт        Новости         36
Технологии   Спорт           31
Спорт        Технологии      22
             Развлечения     19
Развлечения  Спорт           16
dtype: int64

=== Новости -> Технологии: 377 ===

Текст: сша подарить гватемала бронемашина jeep сша безвозмездно передать министерство оборона гватемала бронеавтомобиль jeep общий стоимость миллион доллар это сообщать infodefensa бронемашина поступить на вооружение межведомственный тактический группа текума умать автомобиль среди прочее планироваться использовать для патрулирование приграничный территория ранее март сша безвозмездно передать тактически
Истина: Новости | Предсказано: Технологии

Текст: перу начаться

Ансамбль

In [6]:
import gc
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Обучаем обе модели на train
print("Обучаю LinearSVC...")
svc = LinearSVC(
    C=best_C, class_weight="balanced", dual="auto",
    random_state=RANDOM_STATE, max_iter=15_000,
)
svc.fit(X_tr_vec, y_tr)
gc.collect()

print("Обучаю LogisticRegression...")
lr = LogisticRegression(
    C=best_C_lr, class_weight="balanced", solver="lbfgs",
    max_iter=2_000, tol=1e-4,
    random_state=RANDOM_STATE,
)
lr.fit(X_tr_vec, y_tr)
gc.collect()

assert np.array_equal(svc.classes_, lr.classes_), "Порядок классов различается!"
classes = svc.classes_


def softmax(z, axis=1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)


# Подбор весов на val
svc_val = softmax(svc.decision_function(X_val_vec))
lr_val = lr.predict_proba(X_val_vec)

print("\nПодбор весов ансамбля на val...")
best = (-1, None)
for w_svc in [0.0, 0.2, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0]:
    for w_lr in [0.0, 0.2, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0]:
        if w_svc == 0 and w_lr == 0:
            continue
        w = np.array([w_svc, w_lr]); w /= w.sum()
        blend = w[0] * svc_val + w[1] * lr_val
        pred = classes[blend.argmax(axis=1)]
        score = f1_score(y_val, pred, average="macro")
        if score > best[0]:
            best = (score, (w_svc, w_lr))

print(f"Лучшие веса [svc, lr]: {best[1]}")
print(f"Val macro F1 (ансамбль): {best[0]:.4f}")

svc_val_score = f1_score(y_val, classes[svc_val.argmax(axis=1)], average="macro")
lr_val_score = f1_score(y_val, classes[lr_val.argmax(axis=1)], average="macro")
print(f"Val macro F1 (только SVC): {svc_val_score:.4f}")
print(f"Val macro F1 (только LR):  {lr_val_score:.4f}")

# Выбор финального предиктора
use_ensemble = best[0] > max(svc_val_score, lr_val_score) + 1e-4
if not use_ensemble:
    print("\nАнсамбль не лучше лучшей одиночной модели — на test берём одиночную.")
    if lr_val_score >= svc_val_score:
        pred_final = lr.predict(X_test_vec)
        label = "LogReg (одиночная)"
    else:
        pred_final = svc.predict(X_test_vec)
        label = "LinearSVC (одиночная)"
else:
    print("\nАнсамбль побеждает на val — используем его на test.")
    svc_test = softmax(svc.decision_function(X_test_vec))
    lr_test = lr.predict_proba(X_test_vec)
    w = np.array(best[1]); w /= w.sum()
    blend = w[0] * svc_test + w[1] * lr_test
    pred_final = classes[blend.argmax(axis=1)]
    label = f"Ансамбль SVC+LR (w={tuple(best[1])})"

# Финальная оценка на test
print(f"\nФИНАЛ: {label}")
print("Accuracy:", round(accuracy_score(y_test, pred_final), 4))
print("Macro F1:", round(f1_score(y_test, pred_final, average="macro"), 4))
print("Weighted F1:", round(f1_score(y_test, pred_final, average="weighted"), 4))
print()
print(classification_report(y_test, pred_final, zero_division=0))
gc.collect()

Обучаю LinearSVC...
Обучаю LogisticRegression...

Подбор весов ансамбля на val...
Лучшие веса [svc, lr]: (0.2, 0.0)
Val macro F1 (ансамбль): 0.9388
Val macro F1 (только SVC): 0.9388
Val macro F1 (только LR):  0.9361

Ансамбль не лучше лучшей одиночной модели — на test берём одиночную.

ФИНАЛ: LinearSVC (одиночная)
Accuracy: 0.9399
Macro F1: 0.9398
Weighted F1: 0.9399

              precision    recall  f1-score   support

     Новости       0.91      0.91      0.91      5996
 Развлечения       0.95      0.96      0.95      6000
       Спорт       0.99      0.99      0.99      5999
  Технологии       0.92      0.91      0.91      5991

    accuracy                           0.94     23986
   macro avg       0.94      0.94      0.94     23986
weighted avg       0.94      0.94      0.94     23986



0

## 1. Предобработка

- Отбор рубрик и маппинг 24 исходных в 4 целевые темы:
  - Развлечения = Культура, Ценности, Культпросвет
  - Спорт = Спорт
  - Технологии = Наука и техника, Интернет и СМИ
  - Новости = Россия, Мир, Бывший СССР, Экономика, Бизнес и др.
- Балансировка: до 30 000 записей на класс.
- Очистка: удаление HTML, URL, email; приведение к нижнему регистру; снятие пунктуации; обрезка до 1 200 символов.
- Лемматизация через pymorphy3 с кешем лемм (параллельно joblib.Parallel с threading).
- Удаление дубликатов по text_clean до сплита.
- Сплит: 80% train / 20% test, внутри train 85% / 15% (train/val).

## 2. Метод классификации

FeatureUnion(word TF-IDF 1–2 + char_wb TF-IDF 3–5) + LinearSVC / LogisticRegression.

Подбор C из набора {0.03, 0.1, 0.3, 1, 3, 10} — на отдельной валидационной выборке через f1_macro, без использования GridSearchCV на уже посчитанной матрице. Это устраняет утечку признаков: словарь и IDF обучаются только на X_tr

**Обоснование.** TF-IDF эффективен на разреженных текстовых данных и не требует GPU. Word + char n-grams покрывают лексику и морфологию без контекстных эмбеддингов. LinearSVC быстр, устойчив на многоклассовой задаче и стабильно даёт высокий результат. class_weight=balanced компенсирует дисбаланс. Сведение 24 рубрик в 4 темы устраняет главный источник ошибок — пересечение подкатегорий.

## 3. Результаты

| Модель | Accuracy | Macro F1 |
|---|---|---|
| LinearSVC (C=0.3) | 0.9399 | 0.9398 |
| LogisticRegression (C=3.0) | 0.9357 | 0.9361 |
| Ансамбль SVC+LR | не лучше одиночной | — |


## 4. Анализ ошибок

Ошибок: 1 441 из 23 986 (6.01%). Основные смешения:

- Новости / Технологии — 749 (52% всех ошибок): IT-компании, телеком, кибербезопасность в новостной повестке.
- Новости / Развлечения — 281: новости шоу-бизнеса и культурных событий с политическим контекстом.
- Технологии / Развлечения — 248: интернет-мемы, блогеры, стриминг.
- Новости / Спорт — 75: спортивная политика, допинговые скандалы.

Причины: пересечение тематик, короткие тексты без явных маркеров, дисбаланс сложности между классами. 
Возможные улучшения: Использование поля tags как дополнительного признака(теги сильно коррелируют с рубрикой и в исходном датасете заполнены), расширение обучающей выборки.

## 5. Итог

Пайплайн классификации новостных текстов без трансформеров достигает точности ≥ 0.94 на отложенной тестовой выборке. Ключевые компоненты: лемматизация pymorphy3, свёртка 24 рубрик в 4 темы, word+char TF-IDF, подбор C на отдельной val-выборке, удаление дубликатов до сплита.